# GOV-01 Data Preprocessing

This notebook prepares the duplicate-free road-image split for later modeling. It does **not** train a model or report model metrics.

The model predicts pothole presence only; it does not assess danger, size, severity, or repair priority.

## Before running

The clean split must already exist at `data/processed/clean_split/`. In Google Colab, first obtain the raw Kaggle data and run `src/build_clean_split.py`; raw and derived images are intentionally not stored in GitHub.

In [ ]:
from pathlib import Path
import tensorflow as tf

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = Path('data/processed/clean_split')
CLASS_NAMES = ['Normal', 'Pothole']

tf.keras.utils.set_random_seed(SEED)
print('TensorFlow version:', tf.__version__)
print('Clean split path:', DATA_DIR.resolve())

In [ ]:
required_splits = ['train', 'validation', 'test']
for split in required_splits:
    split_path = DATA_DIR / split
    if not split_path.is_dir():
        raise FileNotFoundError(f'Missing clean split folder: {split_path}')
    available_classes = sorted(path.name for path in split_path.iterdir() if path.is_dir())
    if available_classes != CLASS_NAMES:
        raise ValueError(f'{split} has classes {available_classes}; expected {CLASS_NAMES}')
print('Clean split folders and class names verified.')

In [ ]:
def load_split(split_name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / split_name,
        labels='inferred',
        label_mode='binary',
        class_names=CLASS_NAMES,
        color_mode='rgb',
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED if shuffle else None,
    )

train_raw = load_split('train', shuffle=True)
validation_raw = load_split('validation', shuffle=False)
test_raw = load_split('test', shuffle=False)
print('Image datasets loaded.')

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.10),
], name='training_augmentation')

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.map(
    lambda images, labels: (augmentation(images, training=True), labels),
    num_parallel_calls=AUTOTUNE,
).prefetch(AUTOTUNE)
validation_ds = validation_raw.prefetch(AUTOTUNE)
test_ds = test_raw.prefetch(AUTOTUNE)

print('Training augmentation is enabled only for train_ds.')
print('Validation and test datasets receive no random augmentation.')

## Preprocessing boundary

- Images are resized to 224×224 RGB by the loader.
- Random flip, rotation, and zoom are applied only to training images.
- Pixel scaling is model-specific and will be placed inside each model: `Rescaling(1/255)` for the baseline CNN and `mobilenet_v2.preprocess_input` for MobileNetV2.
- No preprocessing step learns statistics from validation or test images.